# MI-GAN Fine-Tuning: Fix Boundary Box Artifact

This notebook fine-tunes MI-GAN to eliminate the visible rectangular boundary
where generated pixels meet reconstructed pixels in the inpainting output.

**The problem:** MI-GAN has no loss term forcing it to perfectly reconstruct
known (non-masked) pixels. It slightly alters them, creating a brightness/color
gap at the mask boundary.

**The fix:** Two new loss terms:
- **Identity loss** — forces known pixels to match the original exactly
- **Boundary loss** — extra penalty on pixels near the mask edge

**How to use this notebook:**
1. Make sure GPU runtime is enabled: Runtime > Change runtime type > T4 GPU
2. Run cells top to bottom
3. Cell 5 is the quick proof — runs 1K steps, shows before/after
4. Only run Cell 6 (full training) after Cell 5 looks good
5. Cell 8 exports the model for Unity

## Cell 1: Setup — Install Dependencies & Clone Repo

In [ ]:
# Check GPU is available
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU')

# Install dependencies
!pip install -q gdown lpips coremltools Pillow scikit-image

# Clone our repo
import os
if not os.path.exists('/content/migan-finetune'):
    !git clone https://github.com/joscraw/migan-finetune.git /content/migan-finetune
else:
    print('Repo already cloned')

os.chdir('/content/migan-finetune')
print(f'Working directory: {os.getcwd()}')
!ls

## Cell 2: Download Pretrained Weights & Export Inference Model

In [ ]:
import gdown
import pickle
import sys

sys.path.insert(0, '/content/migan-finetune')

# Download the pretrained MI-GAN 512 Places2 checkpoint
os.makedirs('pretrained', exist_ok=True)
pkl_path = 'pretrained/migan_512_places2.pkl'

if not os.path.exists(pkl_path):
    print('Downloading pretrained MI-GAN weights...')
    # MI-GAN Places2 512 model from the official Google Drive
    gdown.download(
        'https://drive.google.com/uc?id=1L1AkQbEbKjFMmKxHp3xjkPfqSh0YmMf',
        pkl_path,
        quiet=False
    )
    print(f'Downloaded to {pkl_path}')
else:
    print(f'Pretrained weights already at {pkl_path}')

# Export to inference model format
# The .pkl contains the training model with reparameterized weights.
# We need to convert it to the clean inference model format.
print('\nLoading training model and exporting to inference format...')

from lib.model_zoo.migan_inference import Generator
from scripts.export_inference_model import copy_weights

# Load training model from pickle
with open(pkl_path, 'rb') as f:
    resume_data = pickle.Unpickler(f).load()
source_G = resume_data['G_ema'].eval().cpu()

# Create inference model and copy weights
inference_model = Generator(resolution=512)
copy_weights(source_G, inference_model, resolution=512)

# Save inference model state_dict
inference_path = 'pretrained/migan_512_inference.pt'
torch.save(inference_model.state_dict(), inference_path)
print(f'Inference model saved to {inference_path}')

# Verify
param_count = sum(p.numel() for p in inference_model.parameters())
print(f'Parameters: {param_count:,}')

# Clean up training model to save memory
del source_G, resume_data
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('Done!')

## Cell 3: Download Training Images & Load Model

In [ ]:
import urllib.request
import tarfile

# Download Places365 validation images (256x256, will be upscaled to 512)
# This is a ~1.8GB download — runs fast on Google's network
images_dir = '/content/places365_val'

if not os.path.exists(images_dir):
    print('Downloading Places365 validation set (~1.8GB)...')
    print('This may take a few minutes on Colab...')
    url = 'http://data.csail.mit.edu/places/places365/val_256.tar'
    tar_path = '/content/val_256.tar'

    if not os.path.exists(tar_path):
        urllib.request.urlretrieve(url, tar_path)
        print(f'Downloaded to {tar_path}')

    print('Extracting...')
    with tarfile.open(tar_path) as tar:
        tar.extractall('/content')

    # Places365 val extracts to /content/val_256/
    if os.path.exists('/content/val_256'):
        os.rename('/content/val_256', images_dir)

    # Clean up tar
    if os.path.exists(tar_path):
        os.remove(tar_path)

    print(f'Extracted to {images_dir}')
else:
    print(f'Images already at {images_dir}')

# Count images
num_images = sum(1 for f in os.listdir(images_dir) if f.endswith('.jpg'))
print(f'Training images: {num_images}')

# Load the pretrained inference model
from finetune.train import load_pretrained

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = load_pretrained('pretrained/migan_512_inference.pt', resolution=512, device=device)
print(f'Model loaded on {device}')

## Cell 4: Measure the Boundary Gap (BEFORE fine-tuning)

This cell runs the pretrained model on test images and measures the
brightness gap at the mask boundary. This is our baseline — the number
we want to drive toward zero.

In [ ]:
from finetune.dataset import InpaintingDataset
from finetune.train import measure_boundary_gap, save_comparison
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Create dataset
dataset = InpaintingDataset(images_dir, resolution=512, hole_range=(0.15, 0.45))

# Measure boundary gap on 20 test samples
print('Measuring boundary gap (BEFORE fine-tuning)...')
test_images = []
test_masks = []
for i in range(20):
    img, mask = dataset[i]
    test_images.append(img)
    test_masks.append(mask)

gap_before = measure_boundary_gap(model, test_images, test_masks, device)
print(f'\n>>> BOUNDARY GAP (BEFORE): {gap_before:.6f}')
print(f'    (This is the average brightness difference across the mask edge)')
print(f'    (Lower = better. 0 = no visible boundary.)')

# Save visual samples
os.makedirs('/content/results/before', exist_ok=True)
for i in range(5):
    save_comparison(model, test_images[i], test_masks[i],
                    f'/content/results/before/sample_{i}.png', device)

# Display samples
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for i in range(min(5, len(test_images))):
    row = i // 3
    col = i % 3
    if i < 5:
        img = Image.open(f'/content/results/before/sample_{i}.png')
        if i < 3:
            axes[0, i].imshow(img)
            axes[0, i].set_title(f'Sample {i}: masked | full output | composited')
            axes[0, i].axis('off')
        elif i < 5:
            axes[1, i-3].imshow(img)
            axes[1, i-3].set_title(f'Sample {i}: masked | full output | composited')
            axes[1, i-3].axis('off')

# Hide unused axes
axes[1, 2].axis('off')
plt.suptitle(f'BEFORE Fine-Tuning — Boundary Gap: {gap_before:.6f}', fontsize=16)
plt.tight_layout()
plt.show()

## Cell 5: Quick Proof — Fine-tune 1,000 Steps

This is the proof-of-concept. Run 1K steps and check if the boundary
gap is shrinking. If it is, continue to Cell 6 for full training.
If not, adjust the loss weights and re-run this cell.

In [ ]:
from finetune.train import train

# Reload fresh pretrained model (so we can re-run this cell)
model = load_pretrained('pretrained/migan_512_inference.pt', resolution=512, device=device)

# Quick proof: 1000 steps
model = train(
    model=model,
    dataset=dataset,
    output_dir='/content/results/proof',
    num_steps=1000,
    batch_size=4,
    lr=1e-4,
    lambda_reconstruction=1.0,
    lambda_identity=10.0,   # Strong — we REALLY want known pixels preserved
    lambda_boundary=5.0,    # Medium-strong — reduce the seam
    lambda_perceptual=0.1,  # Light — maintain structural quality
    use_perceptual=True,
    save_every=250,
    log_every=25,
    device=device,
)

# Measure boundary gap AFTER quick proof
print('\nMeasuring boundary gap (AFTER 1K steps)...')
gap_after_proof = measure_boundary_gap(model, test_images, test_masks, device)
print(f'\n>>> BOUNDARY GAP BEFORE: {gap_before:.6f}')
print(f'>>> BOUNDARY GAP AFTER:  {gap_after_proof:.6f}')
print(f'>>> IMPROVEMENT:         {(1 - gap_after_proof/gap_before)*100:.1f}%')

# Save and display AFTER samples
os.makedirs('/content/results/after_proof', exist_ok=True)
for i in range(5):
    save_comparison(model, test_images[i], test_masks[i],
                    f'/content/results/after_proof/sample_{i}.png', device)

# Side-by-side comparison: BEFORE vs AFTER
fig, axes = plt.subplots(2, 2, figsize=(20, 12))
for i in range(2):
    before_img = Image.open(f'/content/results/before/sample_{i}.png')
    after_img = Image.open(f'/content/results/after_proof/sample_{i}.png')
    axes[i, 0].imshow(before_img)
    axes[i, 0].set_title(f'BEFORE (gap: {gap_before:.6f})', fontsize=14)
    axes[i, 0].axis('off')
    axes[i, 1].imshow(after_img)
    axes[i, 1].set_title(f'AFTER 1K steps (gap: {gap_after_proof:.6f})', fontsize=14)
    axes[i, 1].axis('off')

plt.suptitle(f'Quick Proof: {(1 - gap_after_proof/gap_before)*100:.1f}% improvement', fontsize=18)
plt.tight_layout()
plt.show()

print('\n--- DECISION POINT ---')
print('If the gap is shrinking and images look good: proceed to Cell 6 (full training)')
print('If not improving: adjust lambda values above and re-run this cell')
print('If images are degrading (blurry): reduce lambda_identity or increase lambda_perceptual')

## Cell 6: Full Training (50K steps)

**Only run this after Cell 5 shows improvement.**

This takes ~1-2 hours on a T4 GPU, less on an A100.

In [ ]:
# Reload fresh pretrained model for full training
model = load_pretrained('pretrained/migan_512_inference.pt', resolution=512, device=device)

# Full training: 50K steps
# Adjust these values based on what worked in Cell 5
model = train(
    model=model,
    dataset=dataset,
    output_dir='/content/results/full',
    num_steps=50000,
    batch_size=4,
    lr=1e-4,
    lambda_reconstruction=1.0,
    lambda_identity=10.0,
    lambda_boundary=5.0,
    lambda_perceptual=0.1,
    use_perceptual=True,
    save_every=5000,
    log_every=100,
    device=device,
)

print('\nFull training complete!')

## Cell 7: Final Evaluation

In [ ]:
# Measure final boundary gap
print('Measuring boundary gap (FINAL)...')
gap_final = measure_boundary_gap(model, test_images, test_masks, device)

print(f'\n{"="*50}')
print(f'RESULTS')
print(f'{"="*50}')
print(f'Boundary gap BEFORE:  {gap_before:.6f}')
print(f'Boundary gap AFTER:   {gap_final:.6f}')
print(f'Improvement:          {(1 - gap_final/gap_before)*100:.1f}%')
print(f'{"="*50}')

# Save final comparison images
os.makedirs('/content/results/final', exist_ok=True)
for i in range(10):
    if i < len(test_images):
        save_comparison(model, test_images[i], test_masks[i],
                        f'/content/results/final/sample_{i}.png', device)

# Display before vs after
fig, axes = plt.subplots(3, 2, figsize=(20, 18))
for i in range(3):
    before_img = Image.open(f'/content/results/before/sample_{i}.png')
    after_img = Image.open(f'/content/results/final/sample_{i}.png')
    axes[i, 0].imshow(before_img)
    axes[i, 0].set_title('BEFORE', fontsize=14)
    axes[i, 0].axis('off')
    axes[i, 1].imshow(after_img)
    axes[i, 1].set_title('AFTER (fine-tuned)', fontsize=14)
    axes[i, 1].axis('off')

plt.suptitle(f'Final Results — Gap: {gap_before:.6f} → {gap_final:.6f} ({(1 - gap_final/gap_before)*100:.1f}% improvement)', fontsize=16)
plt.tight_layout()
plt.show()

## Cell 8: Export to Core ML

Converts the fine-tuned model to Core ML format.
Download the `.mlpackage` file, then compile it on your Mac with:

```bash
xcrun coremlcompiler compile migan_finetuned.mlpackage .
```

Then copy the resulting `migan_finetuned.mlmodelc` to:
`ExhibitXR/Assets/StreamingAssets/migan_512.mlmodelc`

In [ ]:
from finetune.convert_coreml import convert_to_coreml

# Convert the fine-tuned model
finetuned_weights = '/content/results/full/migan_finetuned.pt'
coreml_output = '/content/results/migan_finetuned.mlpackage'

convert_to_coreml(finetuned_weights, coreml_output, resolution=512)

print('\n--- NEXT STEPS ---')
print('1. Download migan_finetuned.mlpackage from the Files panel (left sidebar)')
print('2. On your Mac, compile it:')
print('   xcrun coremlcompiler compile migan_finetuned.mlpackage .')
print('3. Copy migan_finetuned.mlmodelc to ExhibitXR/Assets/StreamingAssets/migan_512.mlmodelc')
print('4. Build and test on device!')

## Cell 9: Download Files

Run this to zip and download everything.

In [ ]:
import shutil

# Zip the results
print('Zipping results...')
shutil.make_archive('/content/migan_finetuned_results', 'zip', '/content/results')
print(f'Zip size: {os.path.getsize("/content/migan_finetuned_results.zip") / 1e6:.1f} MB')

# Also zip just the Core ML model
if os.path.exists(coreml_output):
    shutil.make_archive('/content/migan_finetuned_coreml', 'zip', coreml_output)
    print(f'Core ML zip: {os.path.getsize("/content/migan_finetuned_coreml.zip") / 1e6:.1f} MB')

print('\nDownload from the Files panel (folder icon in left sidebar):')
print('  /content/migan_finetuned_results.zip  — all results + checkpoints + samples')
print('  /content/migan_finetuned_coreml.zip   — just the Core ML model')

# Auto-download if running in Colab
try:
    from google.colab import files
    files.download('/content/migan_finetuned_coreml.zip')
except:
    print('\n(Auto-download not available — use the Files panel instead)')